# 10. MoE, routing and MTP — DeepSeek-V4-Flash and Kimi K3

Only tensor widths, vocabulary size, batch and sequence length are reduced. Structural routing constants are preserved.

DeepSeek-V4-Flash:

- 256 routed experts + 1 shared expert
- top-6 routing
- first 3 main-model MoE layers use a frozen token-id → expert-id lookup table
- later layers select with `sqrt(softplus(logit)) + correction_bias`
- mixture weights come from the **unbiased** affinities and are renormalized
- clamped SwiGLU with limit 10
- one MTP depth, with shared token embedding / LM head, RMSNorm fusion and one full causal block

Kimi K3:

- 896 routed experts + 2 shared experts
- top-16 sigmoid routing
- Stable LatentMoE and SiTU-GLU
- Quantile Balancing updates routing bias only; the bias changes selection, not output weights.

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
device = torch.device("cpu")
torch.set_num_threads(min(2, torch.get_num_threads()))
print("device:", device)

## 1. Batched expert primitives

Experts are stored as batched parameter tensors. This keeps the real expert counts without allocating hundreds of Python modules per layer.

In [ ]:
class BatchedSwiGLUExperts(nn.Module):
    def __init__(
        self,
        expert_count,
        input_dim,
        intermediate_dim,
        output_dim,
        clamp_limit=None,
    ):
        super().__init__()
        self.expert_count = expert_count
        self.clamp_limit = clamp_limit

        scale = 0.02
        self.gate_weight = nn.Parameter(
            scale * torch.randn(
                expert_count,
                input_dim,
                intermediate_dim,
            )
        )
        self.value_weight = nn.Parameter(
            scale * torch.randn(
                expert_count,
                input_dim,
                intermediate_dim,
            )
        )
        self.output_weight = nn.Parameter(
            scale * torch.randn(
                expert_count,
                intermediate_dim,
                output_dim,
            )
        )

    def selected_forward(self, hidden, expert_ids):
        gate_weight = self.gate_weight[expert_ids]
        value_weight = self.value_weight[expert_ids]
        output_weight = self.output_weight[expert_ids]

        gate_pre = torch.einsum(
            "bti,btkif->btkf",
            hidden,
            gate_weight,
        )
        value_pre = torch.einsum(
            "bti,btkif->btkf",
            hidden,
            value_weight,
        )

        if self.clamp_limit is not None:
            gate_pre = gate_pre.clamp(max=self.clamp_limit)
            value_pre = value_pre.clamp(
                -self.clamp_limit,
                self.clamp_limit,
            )

        intermediate = F.silu(gate_pre) * value_pre
        return torch.einsum(
            "btkf,btkfo->btko",
            intermediate,
            output_weight,
        )


def situ_gate(x, beta=4.0):
    return beta * torch.tanh(x / beta) * torch.sigmoid(x)


def situ_value(x, beta=25.0):
    return beta * torch.tanh(x / beta)


class BatchedSiTUExperts(nn.Module):
    def __init__(
        self,
        expert_count,
        input_dim,
        intermediate_dim,
        output_dim,
    ):
        super().__init__()
        self.expert_count = expert_count

        scale = 0.02
        self.gate_weight = nn.Parameter(
            scale * torch.randn(
                expert_count,
                input_dim,
                intermediate_dim,
            )
        )
        self.value_weight = nn.Parameter(
            scale * torch.randn(
                expert_count,
                input_dim,
                intermediate_dim,
            )
        )
        self.output_weight = nn.Parameter(
            scale * torch.randn(
                expert_count,
                intermediate_dim,
                output_dim,
            )
        )

    def selected_forward(self, hidden, expert_ids):
        gate_weight = self.gate_weight[expert_ids]
        value_weight = self.value_weight[expert_ids]
        output_weight = self.output_weight[expert_ids]

        gate_pre = torch.einsum(
            "bti,btkif->btkf",
            hidden,
            gate_weight,
        )
        value_pre = torch.einsum(
            "bti,btkif->btkf",
            hidden,
            value_weight,
        )
        intermediate = (
            situ_gate(gate_pre)
            * situ_value(value_pre)
        )
        return torch.einsum(
            "btkf,btkfo->btko",
            intermediate,
            output_weight,
        )

## 2. DeepSeek-V4 gate and frozen hash-table routing

The architecture does **not** invent a modulo hash rule. Hash layers receive the frozen `tid2eid` table as checkpoint data. The small table below is only a test fixture for the reduced vocabulary.

In [ ]:
class DeepSeekV4Gate(nn.Module):
    def __init__(
        self,
        model_dim=8,
        experts=256,
        top_k=6,
        token_to_expert=None,
    ):
        super().__init__()

        assert experts == 256
        assert top_k == 6

        self.experts = experts
        self.top_k = top_k
        self.weight = nn.Parameter(
            torch.randn(experts, model_dim) * 0.02
        )
        self.correction_bias = nn.Parameter(
            torch.zeros(experts),
            requires_grad=False,
        )

        if token_to_expert is None:
            self.register_buffer(
                "token_to_expert",
                None,
            )
        else:
            if token_to_expert.size(-1) != top_k:
                raise ValueError("tid2eid must store exactly top_k experts")
            self.register_buffer(
                "token_to_expert",
                token_to_expert.to(torch.long),
            )

    @property
    def uses_hash_routing(self):
        return self.token_to_expert is not None

    def forward(self, hidden, token_ids=None):
        logits = F.linear(
            hidden.float(),
            self.weight.float(),
        )
        affinity = torch.sqrt(F.softplus(logits))

        if self.uses_hash_routing:
            if token_ids is None:
                raise ValueError(
                    "hash-routed V4 layer requires token ids"
                )
            expert_ids = self.token_to_expert[token_ids]
        else:
            selection_score = affinity + self.correction_bias
            expert_ids = selection_score.topk(
                self.top_k,
                dim=-1,
            ).indices

        selected_affinity = affinity.gather(-1, expert_ids)
        weights = selected_affinity / selected_affinity.sum(
            dim=-1,
            keepdim=True,
        ).clamp_min(1e-8)
        return weights.to(hidden.dtype), expert_ids, affinity


class DeepSeekV4MoE(nn.Module):
    def __init__(
        self,
        model_dim=8,
        intermediate_dim=8,
        token_to_expert=None,
    ):
        super().__init__()

        self.experts = 256
        self.top_k = 6
        self.shared_expert_count = 1

        self.gate = DeepSeekV4Gate(
            model_dim=model_dim,
            experts=self.experts,
            top_k=self.top_k,
            token_to_expert=token_to_expert,
        )
        self.routed = BatchedSwiGLUExperts(
            expert_count=self.experts,
            input_dim=model_dim,
            intermediate_dim=intermediate_dim,
            output_dim=model_dim,
            clamp_limit=10.0,
        )
        self.shared = BatchedSwiGLUExperts(
            expert_count=1,
            input_dim=model_dim,
            intermediate_dim=intermediate_dim,
            output_dim=model_dim,
            clamp_limit=10.0,
        )

    def forward(self, hidden, token_ids=None):
        weights, expert_ids, affinity = self.gate(
            hidden,
            token_ids,
        )
        routed = self.routed.selected_forward(
            hidden,
            expert_ids,
        )
        routed = (
            routed * weights[..., None]
        ).sum(dim=2)

        shared_ids = torch.zeros(
            hidden.size(0),
            hidden.size(1),
            1,
            dtype=torch.long,
            device=hidden.device,
        )
        shared = self.shared.selected_forward(
            hidden,
            shared_ids,
        ).squeeze(2)
        return routed + shared, expert_ids, affinity


# Checkpoint-data fixture: explicit table, never generated by a hash formula.
fixture_tid2eid = torch.tensor(
    [
        [3, 17, 31, 55, 89, 144],
        [2, 19, 37, 61, 101, 173],
        [5, 23, 41, 67, 109, 181],
        [7, 29, 43, 71, 127, 191],
        [11, 47, 73, 131, 193, 223],
        [13, 53, 79, 137, 199, 229],
        [17, 59, 83, 139, 211, 233],
        [19, 61, 97, 149, 227, 251],
    ],
    dtype=torch.long,
)

hash_moe = DeepSeekV4MoE(
    token_to_expert=fixture_tid2eid,
).to(device)
learned_moe = DeepSeekV4MoE().to(device)

assert hash_moe.gate.uses_hash_routing
assert not learned_moe.gate.uses_hash_routing
assert hash_moe.experts == 256
assert hash_moe.top_k == 6
assert hash_moe.shared_expert_count == 1

hidden = torch.randn(1, 3, 8, device=device)
token_ids = torch.tensor([[0, 1, 2]], device=device)
_, hash_ids, _ = hash_moe(hidden, token_ids)
_, learned_ids, _ = learned_moe(hidden)

assert torch.equal(hash_ids, fixture_tid2eid[token_ids])
print("hash ids:", hash_ids)
print("learned ids:", learned_ids)

## 3. Kimi K3 Stable LatentMoE and Quantile Balancing

In [ ]:
class KimiStableLatentMoE(nn.Module):
    def __init__(
        self,
        model_dim=8,
        latent_dim=4,
        routed_intermediate=4,
        shared_intermediate=8,
    ):
        super().__init__()

        self.experts = 896
        self.top_k = 16
        self.shared_expert_count = 2

        self.router = nn.Linear(
            model_dim,
            self.experts,
            bias=False,
        )
        self.routing_bias = nn.Parameter(
            torch.zeros(self.experts),
            requires_grad=False,
        )

        self.down = nn.Linear(
            model_dim,
            latent_dim,
            bias=False,
        )
        self.routed = BatchedSiTUExperts(
            expert_count=self.experts,
            input_dim=latent_dim,
            intermediate_dim=routed_intermediate,
            output_dim=latent_dim,
        )
        self.latent_norm = nn.RMSNorm(latent_dim)
        self.up = nn.Linear(
            latent_dim,
            model_dim,
            bias=False,
        )
        self.shared = BatchedSiTUExperts(
            expert_count=self.shared_expert_count,
            input_dim=model_dim,
            intermediate_dim=shared_intermediate,
            output_dim=model_dim,
        )

    def forward(self, hidden):
        raw_score = torch.sigmoid(self.router(hidden))
        expert_ids = (
            raw_score + self.routing_bias
        ).topk(self.top_k, dim=-1).indices

        selected_raw = raw_score.gather(-1, expert_ids)
        weights = selected_raw / selected_raw.sum(
            dim=-1,
            keepdim=True,
        ).clamp_min(1e-8)

        latent = self.down(hidden)
        routed = self.routed.selected_forward(
            latent,
            expert_ids,
        )
        routed = (
            routed * weights[..., None]
        ).sum(dim=2)
        routed = self.up(self.latent_norm(routed))

        shared_ids = torch.arange(
            self.shared_expert_count,
            device=hidden.device,
        ).view(1, 1, -1).expand(
            hidden.size(0),
            hidden.size(1),
            -1,
        )
        shared = self.shared.selected_forward(
            hidden,
            shared_ids,
        ).sum(dim=2)
        return routed + shared, raw_score, expert_ids


@torch.no_grad()
def quantile_balance_update(
    router_scores,
    routing_bias,
    top_k,
):
    expert_count = router_scores.size(-1)
    biased_score = router_scores + routing_bias

    top_k_plus_one = biased_score.topk(
        top_k + 1,
        dim=-1,
    ).values
    alpha = top_k_plus_one[:, top_k]

    margins = router_scores - alpha[:, None]
    target_quantile = 1.0 - top_k / expert_count
    next_bias = -torch.quantile(
        margins,
        target_quantile,
        dim=0,
    )
    next_bias = next_bias - next_bias.mean()
    routing_bias.copy_(next_bias)
    return next_bias


k3_moe = KimiStableLatentMoE().to(device)
assert k3_moe.experts == 896
assert k3_moe.top_k == 16
assert k3_moe.shared_expert_count == 2

k3_hidden = torch.randn(1, 2, 8, device=device)
k3_output, raw_score, expert_ids = k3_moe(k3_hidden)
old_bias = k3_moe.routing_bias.clone()
next_bias = quantile_balance_update(
    raw_score.detach().reshape(-1, 896),
    k3_moe.routing_bias,
    16,
)
assert not torch.equal(old_bias, next_bias)

loss = k3_output.square().mean()
loss.backward()
print("K3 output:", k3_output.shape)
print("K3 selected experts per token:", expert_ids.size(-1))

## 4. V4 MTP depth: shared embedding/head + normalized fusion + one full causal block

V4 retains the V3 MTP chain. With `num_nextn_predict_layers=1`, the main hidden state and the embedding of the shifted token are separately RMS-normalized, concatenated, projected back to model width, passed through one additional transformer block, then scored with the shared LM head.

In [ ]:
class V4MTPAttention(nn.Module):
    def __init__(
        self,
        model_dim=8,
        query_heads=64,
        head_dim=2,
        sliding_window=128,
    ):
        super().__init__()
        assert query_heads == 64
        assert sliding_window == 128

        self.query_heads = query_heads
        self.head_dim = head_dim
        self.sliding_window = sliding_window

        self.q = nn.Linear(
            model_dim,
            query_heads * head_dim,
            bias=False,
        )
        self.kv = nn.Linear(
            model_dim,
            head_dim,
            bias=False,
        )
        self.out = nn.Linear(
            query_heads * head_dim,
            model_dim,
            bias=False,
        )

    def forward(self, hidden):
        batch_size, sequence_length, _ = hidden.shape
        q = self.q(hidden).view(
            batch_size,
            sequence_length,
            self.query_heads,
            self.head_dim,
        ).transpose(1, 2)
        kv = self.kv(hidden)
        k = kv[:, None].expand(
            -1,
            self.query_heads,
            -1,
            -1,
        )
        v = k

        position = torch.arange(sequence_length, device=hidden.device)
        distance = position[:, None] - position[None, :]
        mask = (distance >= 0) & (distance < self.sliding_window)

        attended = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=mask,
        )
        attended = attended.transpose(1, 2).contiguous().flatten(2)
        return self.out(attended)


class V4MTPTransformerBlock(nn.Module):
    def __init__(self, model_dim=8):
        super().__init__()
        self.norm1 = nn.RMSNorm(model_dim)
        self.attention = V4MTPAttention(model_dim=model_dim)
        self.norm2 = nn.RMSNorm(model_dim)
        self.moe = DeepSeekV4MoE(model_dim=model_dim)

    def forward(self, hidden):
        hidden = hidden + self.attention(self.norm1(hidden))
        moe_output, _, _ = self.moe(self.norm2(hidden))
        return hidden + moe_output


class DeepSeekV4MTPDepth(nn.Module):
    def __init__(
        self,
        shared_embedding,
        shared_lm_head,
        model_dim=8,
    ):
        super().__init__()

        self.enorm = nn.RMSNorm(model_dim)
        self.hnorm = nn.RMSNorm(model_dim)
        self.eh_projection = nn.Linear(
            2 * model_dim,
            model_dim,
            bias=False,
        )
        self.transformer_block = V4MTPTransformerBlock(
            model_dim=model_dim,
        )
        self.final_norm = nn.RMSNorm(model_dim)

        self.shared_embedding = shared_embedding
        self.shared_lm_head = shared_lm_head

    def forward(self, hidden, shifted_token_ids):
        embedding = self.shared_embedding(shifted_token_ids)
        fused = self.eh_projection(
            torch.cat(
                [self.hnorm(hidden), self.enorm(embedding)],
                dim=-1,
            )
        )
        mtp_hidden = self.transformer_block(fused)
        return self.shared_lm_head(self.final_norm(mtp_hidden))


vocabulary_size = 32
model_dim = 8
shared_embedding = nn.Embedding(vocabulary_size, model_dim).to(device)
shared_lm_head = nn.Linear(
    model_dim,
    vocabulary_size,
    bias=False,
).to(device)
mtp_depths = nn.ModuleList(
    [
        DeepSeekV4MTPDepth(
            shared_embedding,
            shared_lm_head,
            model_dim=model_dim,
        )
    ]
).to(device)

assert len(mtp_depths) == 1
assert mtp_depths[0].shared_embedding is shared_embedding
assert mtp_depths[0].shared_lm_head is shared_lm_head
assert mtp_depths[0].transformer_block.moe.experts == 256
assert mtp_depths[0].transformer_block.moe.top_k == 6

main_hidden = torch.randn(1, 4, model_dim, device=device)
shifted_ids = torch.randint(0, vocabulary_size, (1, 4), device=device)
mtp_logits = mtp_depths[0](main_hidden, shifted_ids)
print("MTP depth count:", len(mtp_depths))
print("MTP logits:", mtp_logits.shape)

## Structural checklist

Assertions verify that expert counts and top-k values were **not** reduced: V4 = 256/top-6/+1 shared, K3 = 896/top-16/+2 shared. V4 hash routing consumes a frozen lookup table instead of a fabricated modulo shortcut. The MTP path has one depth and shares embedding/head weights as specified by the released V4-Flash configuration.